# Getting started with CivKit in Python

> **Disclaimer.** CivKit results support, and do not replace, the judgement of a qualified engineer.
>
> Everything here runs offline: the `civkit` package makes no network calls.

This notebook lists the calculations, runs one in the three quantity forms, converts units, reads the trace, and
handles errors. It needs only the `civkit` package (`pip install civkit`, or `maturin develop` in `bindings/python`).

In [ ]:
import json

import civkit

print("CivKit", civkit.__version__)
for d in civkit.list():
    print(f"{d['id']:45} {d['title']}")

## What a calculation needs

`civkit.describe` returns the descriptor: inputs with their dimension, default unit and domain, outputs, assumptions,
references. The same information is in each typed function's docstring (`help(civkit.fluids.open_channel.manning_trapezoidal)`).

In [ ]:
d = civkit.describe("fluids.open_channel.manning_trapezoidal")
print(d["title"])
for i in d["inputs"]:
    print(f"  {i['name']:14} {i.get('display_unit') or '(dimensionless)':10} {i['description']}")

## Three ways to give a quantity

A string `"1.5 m"`, a tuple `(1.5, "m")` or `civkit.Q(1.5, "m")`; dimensionless inputs take a plain number. A dimensional
input given as a bare number is refused (`MISSING_UNIT`): CivKit never assumes a unit.

In [ ]:
r = civkit.fluids.open_channel.manning_trapezoidal(
    bottom_width="1.5 m", side_slope=2.0, depth=(0.6, "m"), manning_n=0.013, bed_slope=0.002
)
same = civkit.fluids.open_channel.manning_trapezoidal(
    bottom_width=civkit.Q(1.5, "m"), side_slope=2.0, depth="600 mm", manning_n=0.013, bed_slope=0.002
)
assert r.outputs["discharge"].value == same.outputs["discharge"].value
for name, q in r.outputs.items():
    print(f"{name:18} {q.value:.6g} {q.unit or ''}")

## Units of the results

Ask for output units in the call, or convert a result with `.to()`. Both run the conversion in the engine.

In [ ]:
in_feet = civkit.fluids.open_channel.manning_trapezoidal(
    bottom_width="1.5 m", side_slope=2.0, depth="0.6 m", manning_n=0.013, bed_slope=0.002,
    output_units={"discharge": "L/s", "velocity": "ft/s"},
)
print(in_feet.outputs["discharge"], in_feet.outputs["velocity"])
print(r.outputs["velocity"].to("km/h"))

## The trace

Every result carries the steps that produced it, in SI, so a number can be checked by hand.

In [ ]:
for step in r.trace:
    print(f"{step['step']:40} {step.get('value', ''):>12.6g} {step.get('unit', '')}")

## Errors are data

A refused request raises `civkit.CivKitError` with a code, the input concerned, and a message. Several problems are
reported together, in `.errors`.

In [ ]:
try:
    civkit.fluids.open_channel.manning_trapezoidal(
        bottom_width="1.5 m", side_slope=2.0, depth=0.6, manning_n=0.013, bed_slope=0.002
    )
except civkit.CivKitError as e:
    print(e.code, e.input, "-", e.message)

try:
    civkit.fluids.open_channel.manning_trapezoidal(
        bottom_width="-1 m", side_slope=2.0, depth="0 m", manning_n=0.013, bed_slope=0.002
    )
except civkit.CivKitError as e:
    for err in e.errors:
        print(err["code"], err["input"], "-", err["message"])

## The JSON contract

`to_dict()` is exactly the engine's JSON response, the same one the MCP server and every other interface return.

In [ ]:
print(json.dumps(r.to_dict(), indent=1, ensure_ascii=False)[:800])